# 03 - Baseline Classico (Trilha A) + Objetivo Especifico 1

Objetivo: usar as features agregadas por clipe (media/desvio/percentis de EAR, MAR, yaw, pitch, roll, taxa de deteccao de rosto) para treinar classificadores classicos (RandomForest, XGBoost, SVM) com `GridSearchCV`, comparando estrategias de balanceamento de classes.

Este notebook responde diretamente:
- **Objetivo Geral** (Secao 2 do plano): comparacao com baseline trivial (macro-F1/kappa).
- **Objetivo Especifico 1**: ranking de importancia de features por rotulo.

Pre-requisito: `02_preprocessamento_landmarks.ipynb` executado (arquivos `.parquet` em `datasets/DAiSEE/features/`).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score, balanced_accuracy_score,
    classification_report, confusion_matrix
)
from sklearn.dummy import DummyClassifier
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
ROOT = Path.cwd().parent
FEATURES_DIR = ROOT / "datasets" / "DAiSEE" / "features"
TARGET = "Engagement"  # trocar para Boredom/Confusion/Frustration para repetir a analise (Objetivo 1)

In [ ]:
def load_and_aggregate(split_name):
    df = pd.read_parquet(FEATURES_DIR / f"{split_name}_frame_features.parquet")
    feat_cols = ["ear", "mar", "yaw", "pitch", "roll"]

    agg_funcs = ["mean", "std", "min", "max"]
    grouped = df.groupby("ClipID")

    agg = grouped[feat_cols].agg(agg_funcs)
    agg.columns = ["_".join(c) for c in agg.columns]
    face_rate = grouped["face_detected"].mean().rename("face_detection_rate")
    labels = grouped[["Boredom", "Engagement", "Confusion", "Frustration"]].first()

    out = agg.join(face_rate).join(labels).reset_index()
    return out

train_agg = load_and_aggregate("train")
val_agg = load_and_aggregate("validation")
test_agg = load_and_aggregate("test")

feature_cols = [c for c in train_agg.columns if c not in ["ClipID", "Boredom", "Engagement", "Confusion", "Frustration"]]
print("Numero de features:", len(feature_cols))
train_agg.head()

In [ ]:
X_train, y_train = train_agg[feature_cols].fillna(0), train_agg[TARGET]
X_val, y_val = val_agg[feature_cols].fillna(0), val_agg[TARGET]
X_test, y_test = test_agg[feature_cols].fillna(0), test_agg[TARGET]

print("Distribuicao y_train:", y_train.value_counts(normalize=True).round(3).to_dict())

## Metricas completas obrigatorias (ver Secao 6 do plano)

In [ ]:
def full_report(y_true, y_pred, nome_modelo):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    kappa = cohen_kappa_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)

    print(f"\n===== {nome_modelo} =====")
    print(f"Acuracia: {acc:.4f} | Macro-F1: {macro_f1:.4f} | Kappa: {kappa:.4f} | Balanced Acc: {bal_acc:.4f}")
    print(classification_report(y_true, y_pred, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Matriz de Confusao - {nome_modelo}")
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.show()

    return {"modelo": nome_modelo, "acc": acc, "macro_f1": macro_f1, "kappa": kappa, "balanced_acc": bal_acc}

## Baseline trivial (para validar a hipotese, Secao 2 e 10 do plano)

In [ ]:
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
results = [full_report(y_test, dummy.predict(X_test), "Baseline trivial (classe majoritaria)")]

## Estrategia 1: sem tratamento de desbalanceamento

In [ ]:
rf_param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 5],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    rf_param_grid, scoring="f1_macro", cv=cv, n_jobs=-1,
)
rf_grid.fit(X_train, y_train)
print("Melhores parametros:", rf_grid.best_params_)
results.append(full_report(y_test, rf_grid.best_estimator_.predict(X_test), "RandomForest (sem balanceamento)"))

## Estrategia 2: class_weight='balanced'

In [ ]:
rf_balanced_grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    rf_param_grid, scoring="f1_macro", cv=cv, n_jobs=-1,
)
rf_balanced_grid.fit(X_train, y_train)
best_rf = rf_balanced_grid.best_estimator_
results.append(full_report(y_test, best_rf.predict(X_test), "RandomForest (class_weight=balanced)"))

## Estrategia 3: oversampling com SMOTE (apenas no Train, sem vazamento)

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print("Distribuicao apos SMOTE:", pd.Series(y_train_sm).value_counts().to_dict())

rf_smote = RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=400)
rf_smote.fit(X_train_sm, y_train_sm)
results.append(full_report(y_test, rf_smote.predict(X_test), "RandomForest + SMOTE"))

## XGBoost (com scale/peso por classe) e SVM para comparacao

In [ ]:
sample_weights = y_train.map(y_train.value_counts(normalize=True).rpow(-1))
xgb_model = xgb.XGBClassifier(
    random_state=RANDOM_STATE, n_estimators=400, max_depth=6, learning_rate=0.05,
    eval_metric="mlogloss",
)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
results.append(full_report(y_test, xgb_model.predict(X_test), "XGBoost (sample_weight balanceado)"))

svm_model = SVC(kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE)
svm_model.fit(X_train, y_train)
results.append(full_report(y_test, svm_model.predict(X_test), "SVM (class_weight=balanced)"))

## Objetivo Especifico 1 - Ranking de importancia de features

Repetir para cada rotulo (Boredom, Engagement, Confusion, Frustration) e validar estabilidade do top-5 via CV (ver criterio de mensurabilidade na Secao 2 do plano).

In [ ]:
importances = pd.Series(best_rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(f"Top-10 features para {TARGET}:")
print(importances.head(10))

importances.head(15).plot(kind="barh", figsize=(7, 6))
plt.title(f"Importancia de features (RandomForest) - {TARGET}")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Estabilidade do ranking via K-Fold (top-5 deve se repetir em >= 4/5 folds - criterio do Objetivo 1)
from collections import Counter

top5_per_fold = []
for train_idx, _ in cv.split(X_train, y_train):
    rf_fold = RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=200, class_weight="balanced")
    rf_fold.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
    top5 = pd.Series(rf_fold.feature_importances_, index=feature_cols).sort_values(ascending=False).head(5).index.tolist()
    top5_per_fold.append(tuple(top5))

feature_stability = Counter([f for fold in top5_per_fold for f in fold])
print("Frequencia de cada feature no top-5 (max=5 folds):")
print(pd.Series(feature_stability).sort_values(ascending=False))

## Consolidacao dos resultados desta trilha

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv(FEATURES_DIR / "resultados_trilha_a_classico.csv", index=False)
results_df

## Checklist de saida
- [ ] Baseline trivial e ao menos 4 variantes de balanceamento comparadas com metricas completas
- [ ] Ranking de importancia de features gerado e estabilidade avaliada (Objetivo Especifico 1)
- [ ] `resultados_trilha_a_classico.csv` salvo para consolidacao no Notebook 05

Proximo passo: `04_modelo_temporal_dl.ipynb`.